# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset (ordered logistic regression outputs for knowledge adoption predictors in rangeland management) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into *record sets*, each with fields (columns, features) identified by their `@id`. Let's enumerate all available record sets and their fields by `@id`.

In [ ]:
# List available record sets and their corresponding fields, all by @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in this dataset schema.\n")
else:
    for record_set in record_sets:
        print(f"Record set: {record_set['@id']}")
        if 'fields' in record_set and record_set['fields']:
            for field in record_set['fields']:
                print(f"    Field: {field['@id']}")
        else:
            print("    (No fields listed)")

Let's attempt to display a small sample from the first record set, if available.
To do this, we need to reference record sets by their `@id` only.

In [ ]:
# Show first few records from the first record set (by @id), if present
if record_sets:
    first_recordset_id = record_sets[0]['@id']
    print(f"First record set @id: {first_recordset_id}")
    print("Sample record:")
    for i, record in enumerate(dataset.records(record_set=first_recordset_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets with data to display.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame.

Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set

dataframes = {}
all_record_set_ids = []
for record_set in record_sets:
    rec_id = record_set['@id']
    all_record_set_ids.append(rec_id)
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded records for RecordSet @id: {rec_id}. Shape: {df.shape}")
    else:
        print(f"No records found for RecordSet @id: {rec_id}.")

# For demonstration, select the first available DataFrame
if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set {selected_rs_id}:")
    print(dataframes[selected_rs_id].columns.tolist())
    display(dataframes[selected_rs_id].head())
else:
    print("No record sets with extractable tabular data found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps to the loaded DataFrame: filtering, normalization, grouping.

For demonstration, we will:
- Select a numeric field (e.g., with 'coefficient' or 'log_likelihood' in its name, if present).
- Filter records using a threshold.
- Normalize the numeric field.
- Group by a categorical field if available.

In [ ]:
# Try to automatically select a numeric and a categorical field
import numpy as np

# Pick the DataFrame loaded above
if dataframes:
    df = dataframes[selected_rs_id]
    # Try to find likely numeric and group fields
    numeric_candidates = [col for col in df.columns if (
        'coefficient' in col.lower() or 'value' in col.lower() or 'log_likelihood' in col.lower() or df[col].dtype in [np.float64, np.int64]
    )]
    group_candidates = [col for col in df.columns if (
        'category' in col.lower() or 'group' in col.lower() or 'variable' in col.lower() or df[col].dtype == object
    )]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].dropna().quantile(0.5) # use median as an example threshold

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_candidates:
            group_field = group_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by {group_field} (mean of {numeric_field}):")
                display(grouped_df.head())
    else:
        print("No numeric fields found for EDA. Columns available:")
        print(df.columns.tolist())
else:
    print("No data loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will visualize the distribution of the selected numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    # Histogram for numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, color='orange')
        plt.title(f"Distribution of Normalized {numeric_field}")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel('Count')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load a Croissant-structured dataset and its metadata with `mlcroissant`.
- Explore record sets and their fields using `@id` references.
- Extract data into DataFrames for analysis by referencing record set `@id`.
- Perform basic EDA: filtering, normalization, grouping, and plotting data distributions.

Continue exploring the dataset further to gain deeper insights into knowledge adoption and rangeland management practices in Northern Kenya.